# 🔥 Calories Burned Prediction — Full ML Pipeline

| 항목 | 내용 |
|---|---|
| **환경** | Google Colab (GPU 권장) |
| **모델** | XGBoost · LightGBM · CatBoost · HistGradientBoosting |
| **전략** | KFold CV + RandomizedSearchCV 하이퍼파라미터 최적화 |
| **타겟** | log1p(Calories_Burned) → RMSE 최소화 |

## ⚙️ 0. Colab 환경 설정 & 패키지 설치

In [ ]:
# ── Colab GPU 체크
import subprocess, sys

try:
    gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if 'failed' in gpu_info.stdout.lower() or gpu_info.returncode != 0:
        print('⚠️  GPU 없음 — CPU로 실행합니다. Colab에서 런타임 > GPU로 변경 권장')
        USE_GPU = False
    else:
        print('✅ GPU 사용 가능:')
        print(gpu_info.stdout[:200])
        USE_GPU = True
except FileNotFoundError:
    print('⚠️  nvidia-smi 없음 — CPU 모드')
    USE_GPU = False

# ── 패키지 설치 (Colab에 없는 경우)
!pip install -q xgboost lightgbm catboost optuna koreanize-matplotlib

print('\n✅ 패키지 설치 완료')

## 📦 1. Import & 전역 설정

In [ ]:
import pandas as pd
import numpy as np
import time, warnings, copy, os
warnings.simplefilter('ignore')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from sklearn.model_selection  import KFold, RandomizedSearchCV
from sklearn.preprocessing    import LabelEncoder, OrdinalEncoder
from sklearn.metrics          import mean_squared_error
from sklearn.ensemble         import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

from xgboost  import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

#한글 폰트 (Colab/로컬 공통)
try:
    import koreanize_matplotlib
except ImportError:
    plt.rcParams["font.family"] = "AppleGothic"   # macOS
    plt.rcParams["axes.unicode_minus"] = False

# ─────────────────────────────────────────
#  전역 상수
# ─────────────────────────────────────────
SEED       = 42
FOLDS      = 10          # KFold 분할 수
N_ITER     = 30          # RandomizedSearch 반복 횟수 (모델당)
HP_FOLDS   = 5           # HP 탐색용 내부 CV
CAL_MIN    = 1
CAL_MAX    = 300         # 타겟 클리핑 범위

np.random.seed(SEED)

# ─────────────────────────────────────────
#  다크 테마 시각화 설정
# ─────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0d1117',
    'axes.facecolor'   : '#161b22',
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#c9d1d9',
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'text.color'       : '#e6edf3',
    'grid.color'       : '#21262d',
    'grid.alpha'       : 0.8,
    'axes.grid'        : True,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'figure.titlesize' : 16,
    'figure.titleweight':'bold',
})

# 모델별 색상
MODEL_COLORS = {
    'XGBoost' : '#f97316',
    'LightGBM': '#22c55e',
    'CatBoost': '#3b82f6',
    'HistGB'  : '#a855f7',
    'Ensemble': '#fbbf24',
}

print('✅ 모든 라이브러리 로드 완료')
print(f'   XGBoost  : {XGBRegressor().__class__.__module__.split(".")[0]}')
print(f'   LightGBM : lgb')
print(f'   CatBoost : cb')
print(f'   USE_GPU  : {USE_GPU}')

## 📥 2. 데이터 로드

In [ ]:
# ── Colab이면 Drive 마운트 또는 파일 업로드 옵션
import os

TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'

# Colab 업로드 대안: 파일이 없으면 업로드 UI 표시
if not os.path.exists(TRAIN_PATH):
    try:
        from google.colab import files
        print('파일을 업로드하세요 (train.csv, test.csv)')
        uploaded = files.upload()
    except ImportError:
        print('⚠️  train.csv / test.csv 를 현재 디렉토리에 배치하세요')

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

print(f'✅ Train : {train_raw.shape[0]:,} rows × {train_raw.shape[1]} cols')
print(f'   Test  : {test_raw.shape[0]:,} rows × {test_raw.shape[1]} cols')
print()
print('컬럼 목록:', train_raw.columns.tolist())
train_raw.head()

## 🔍 3. EDA — 탐색적 데이터 분석

In [ ]:
# ─── 기본 통계
print('='*60)
print('  기본 통계')
print('='*60)
display(train_raw.describe(include='all'))

print('\n결측값:', train_raw.isnull().sum().to_dict())
print('중복행 :', train_raw.duplicated().sum())

In [ ]:
# ─── 타겟 분포 (3패널)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('📊 Target Distribution — Calories_Burned')

cal = train_raw['Calories_Burned']

# 원본
axes[0].hist(cal, bins=60, color='#f97316', alpha=0.85, edgecolor='none')
axes[0].axvline(cal.mean(), color='white', ls='--', lw=1.5, label=f'Mean={cal.mean():.1f}')
axes[0].axvline(cal.median(), color='#22c55e', ls='--', lw=1.5, label=f'Median={cal.median():.1f}')
axes[0].set_title('Raw Distribution')
axes[0].set_xlabel('Calories_Burned')
axes[0].legend()

# log1p 변환
log_cal = np.log1p(cal)
axes[1].hist(log_cal, bins=60, color='#22c55e', alpha=0.85, edgecolor='none')
axes[1].axvline(log_cal.mean(), color='white', ls='--', lw=1.5, label=f'Mean={log_cal.mean():.3f}')
axes[1].set_title(f'log1p Distribution  (skew: {cal.skew():.2f} → {log_cal.skew():.2f})')
axes[1].set_xlabel('log1p(Calories_Burned)')
axes[1].legend()

# 성별 비교
for g, c in zip(['M', 'F'], ['#3b82f6', '#ec4899']):
    axes[2].hist(train_raw[train_raw['Gender']==g]['Calories_Burned'],
                 bins=50, alpha=0.6, color=c, label=g, edgecolor='none')
axes[2].set_title('Distribution by Gender')
axes[2].set_xlabel('Calories_Burned')
axes[2].legend(title='Gender')

plt.tight_layout()
plt.savefig('plot_01_target_dist.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'  Skewness  원본: {cal.skew():.4f}  →  log1p: {log_cal.skew():.4f}')
print(f'  Kurtosis  원본: {cal.kurt():.4f}  →  log1p: {log_cal.kurt():.4f}')

In [ ]:
# ─── 수치형 피처 분포
num_cols = ['Exercise_Duration','Body_Temperature(F)','BPM',
            'Height(Feet)','Height(Remainder_Inches)','Weight(lb)','Age']

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle('📊 Numerical Feature Distributions')
axes = axes.flatten()
palette = sns.color_palette('husl', len(num_cols))

for i, col in enumerate(num_cols):
    axes[i].hist(train_raw[col], bins=40, color=palette[i], alpha=0.85, edgecolor='none')
    mu, sd = train_raw[col].mean(), train_raw[col].std()
    axes[i].axvline(mu, color='white', ls='--', lw=1.5)
    axes[i].set_title(f'{col}\nμ={mu:.1f}  σ={sd:.1f}')
    axes[i].set_xlabel(col)

axes[7].hist(train_raw['Calories_Burned'], bins=40, color='#fbbf24', alpha=0.85, edgecolor='none')
axes[7].set_title('Calories_Burned (Target)')
axes[7].set_xlabel('Calories')

plt.tight_layout()
plt.savefig('plot_02_num_distributions.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── 범주형 분석
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
fig.suptitle('📊 Categorical Feature Analysis')

# Weight Status 분포
ws = train_raw['Weight_Status'].value_counts()
axes[0].bar(ws.index, ws.values,
            color=sns.color_palette('cool', len(ws)), edgecolor='none', alpha=0.9)
for i,(k,v) in enumerate(ws.items()):
    axes[0].text(i, v+20, str(v), ha='center', fontsize=10)
axes[0].set_title('Weight_Status Count')

# Gender 분포
g = train_raw['Gender'].value_counts()
axes[1].bar(g.index, g.values, color=['#3b82f6','#ec4899'], edgecolor='none', alpha=0.9)
for i,(k,v) in enumerate(g.items()):
    axes[1].text(i, v+20, str(v), ha='center', fontsize=10)
axes[1].set_title('Gender Count')

# Weight Status별 Calories 박스플롯
order = ['Normal Weight','Overweight','Obese']
data_ws = [train_raw[train_raw['Weight_Status']==s]['Calories_Burned'].values for s in order]
bp = axes[2].boxplot(data_ws, labels=order, patch_artist=True,
                     medianprops={'color':'white','lw':2})
for patch, c in zip(bp['boxes'], ['#22c55e','#f97316','#ef4444']):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[2].set_title('Calories by Weight_Status')
axes[2].set_xticklabels(order, rotation=15)

# Gender별 Calories 바이올린
for i, (g_val, c) in enumerate(zip(['M','F'], ['#3b82f6','#ec4899'])):
    data = train_raw[train_raw['Gender']==g_val]['Calories_Burned']
    vp = axes[3].violinplot(data, positions=[i], showmedians=True)
    for pc in vp['bodies']:
        pc.set_facecolor(c); pc.set_alpha(0.7)
    vp['cmedians'].set_color('white')
axes[3].set_xticks([0,1]); axes[3].set_xticklabels(['M','F'])
axes[3].set_title('Calories Distribution by Gender')

plt.tight_layout()
plt.savefig('plot_03_categorical.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── 피처 vs 타겟 산점도
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle('📊 Feature vs Calories_Burned (Scatter + Correlation)')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sc = axes[i].scatter(
        train_raw[col], train_raw['Calories_Burned'],
        c=train_raw['Calories_Burned'], cmap='plasma',
        alpha=0.35, s=12, edgecolors='none'
    )
    r, p = stats.pearsonr(train_raw[col], train_raw['Calories_Burned'])
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Calories')
    axes[i].set_title(f'{col}   r={r:.3f}')
    # 추세선
    z = np.polyfit(train_raw[col], train_raw['Calories_Burned'], 1)
    p_ = np.poly1d(z)
    xs = np.linspace(train_raw[col].min(), train_raw[col].max(), 100)
    axes[i].plot(xs, p_(xs), 'w--', lw=1.5)

axes[7].set_visible(False)

plt.tight_layout()
plt.savefig('plot_04_scatter.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── 상관관계 히트맵
fig, ax = plt.subplots(figsize=(12, 9))
fig.suptitle('📊 Correlation Heatmap')

corr = train_raw[num_cols + ['Calories_Burned']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    annot=True, fmt='.2f', annot_kws={'size':10},
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='#0d1117',
    cbar_kws={'shrink':0.8}
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig('plot_05_heatmap.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# 상관관계 순위 출력
print('\n타겟과의 상관계수 순위:')
print(corr['Calories_Burned'].drop('Calories_Burned').abs().sort_values(ascending=False).to_string())

## 🔧 4. 전처리 & 피처 엔지니어링

In [ ]:
def build_features(df):
    d = df.copy()

    # ── 단위 변환
    d['Height_in'] = d['Height(Feet)'] * 12 + d['Height(Remainder_Inches)']
    d['Height_cm'] = d['Height_in'] * 2.54
    d['Weight_kg'] = d['Weight(lb)'] * 0.453592
    d['Temp_C']    = (d['Body_Temperature(F)'] - 32) * 5 / 9

    # ── BMI & 파생
    d['BMI']           = d['Weight_kg'] / (d['Height_cm'] / 100) ** 2
    d['Temp_Excess']   = d['Body_Temperature(F)'] - 98.6   # 정상체온 대비 초과
    d['Intensity']     = d['BPM'] * d['Exercise_Duration'] # 운동강도 근사값
    d['Duration_sq']   = d['Exercise_Duration'] ** 2
    d['Age_sq']        = d['Age'] ** 2
    d['BPM_Age']       = d['BPM'] / (d['Age'] + 1)
    d['Weight_Duration']= d['Weight_kg'] * d['Exercise_Duration']
    d['Temp_BPM']      = d['Temp_C'] * d['BPM']

    # ── 교차항 (핵심 피처 6가지 조합)
    key = ['Exercise_Duration','Body_Temperature(F)','BPM','Age','Weight_kg','Height_cm']
    from itertools import combinations
    for f1, f2 in combinations(key, 2):
        d[f'{f1}_x_{f2}'] = d[f1] * d[f2]

    # ── 범주형 인코딩
    d['Gender_enc']      = (d['Gender'] == 'M').astype(np.int8)
    ws_map = {'Normal Weight': 0, 'Overweight': 1, 'Obese': 2}
    d['WeightStatus_enc'] = d['Weight_Status'].map(ws_map).fillna(1).astype(np.int8)

    # ── 불필요 컬럼 제거
    drop = ['ID','Height(Feet)','Height(Remainder_Inches)',
            'Weight(lb)','Gender','Weight_Status']
    if 'Calories_Burned' in d.columns:
        drop.append('Calories_Burned')
    d.drop(columns=[c for c in drop if c in d.columns], inplace=True)

    return d


X     = build_features(train_raw)
y     = np.log1p(train_raw['Calories_Burned'].values)
X_test= build_features(test_raw)

FEATURES = X.columns.tolist()
print(f'✅ 피처 수 : {len(FEATURES)}')
print(f'   피처 목록: {FEATURES}')

In [ ]:
# ─── 엔지니어링 후 상관관계 확인
fe_corr = pd.concat([X, pd.Series(y, name='log_Calories')], axis=1).corr()['log_Calories']
fe_corr = fe_corr.drop('log_Calories').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
fig.suptitle('📊 Feature Correlation with log1p(Calories) — After Engineering')

colors = ['#f97316' if v > 0.7 else '#22c55e' if v > 0.4 else '#8b949e'
          for v in fe_corr.values]
ax.barh(range(len(fe_corr)), fe_corr.values[::-1][::-1], color=colors[::-1], alpha=0.85, edgecolor='none')
ax.set_yticks(range(len(fe_corr)))
ax.set_yticklabels(fe_corr.index, fontsize=9)
ax.set_xlabel('|Pearson r|')
ax.axvline(0.4, color='#22c55e', ls='--', lw=1, alpha=0.7, label='r=0.4')
ax.axvline(0.7, color='#f97316', ls='--', lw=1, alpha=0.7, label='r=0.7')
ax.legend()

plt.tight_layout()
plt.savefig('plot_06_feature_corr.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('\nTop-10 피처:')
print(fe_corr.head(10).to_string())

## 🎛️ 5. 하이퍼파라미터 탐색 (RandomizedSearchCV)

In [ ]:
from scipy.stats import randint, uniform

# ── 탐색 공간 정의
XGB_TREE_METHOD = 'hist'
XGB_DEVICE      = 'cuda' if USE_GPU else 'cpu'

search_configs = {
    'XGBoost': (
        XGBRegressor(
            tree_method=XGB_TREE_METHOD,
            device=XGB_DEVICE,
            eval_metric='rmse',
            random_state=SEED,
            enable_categorical=False
        ),
        {
            'n_estimators'     : [300, 500, 800, 1000, 1500],
            'max_depth'        : [4, 5, 6, 7, 8, 10],
            'learning_rate'    : [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
            'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree' : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'gamma'            : [0, 0.01, 0.05, 0.1, 0.3],
            'min_child_weight' : [1, 2, 3, 5],
            'reg_alpha'        : [0, 0.01, 0.05, 0.1, 0.5],
            'reg_lambda'       : [0.5, 1.0, 1.5, 2.0, 3.0],
        }
    ),
    'LightGBM': (
        LGBMRegressor(
            random_state=SEED,
            n_jobs=-1,
            verbose=-1
        ),
        {
            'n_estimators'     : [300, 500, 800, 1000, 1500],
            'max_depth'        : [-1, 5, 7, 10, 12],
            'learning_rate'    : [0.005, 0.01, 0.02, 0.03, 0.05, 0.08],
            'num_leaves'       : [15, 31, 63, 127, 255],
            'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree' : [0.5, 0.6, 0.7, 0.8, 0.9],
            'reg_alpha'        : [0, 0.01, 0.05, 0.1, 0.5],
            'reg_lambda'       : [0, 0.1, 0.5, 1.0, 2.0],
            'min_child_samples': [5, 10, 20, 30, 50],
        }
    ),
    'CatBoost': (
        CatBoostRegressor(
            random_state=SEED,
            verbose=0,
            task_type='GPU' if USE_GPU else 'CPU'
        ),
        {
            'iterations'       : [300, 500, 800, 1000],
            'depth'            : [4, 5, 6, 7, 8, 10],
            'learning_rate'    : [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
            'l2_leaf_reg'      : [1, 3, 5, 7, 10],
            'bagging_temperature': [0, 0.2, 0.5, 1.0],
            'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
        }
    ),
    'HistGB': (
        HistGradientBoostingRegressor(
            random_state=SEED
        ),
        {
            'max_iter'         : [300, 500, 800, 1000],
            'max_depth'        : [3, 4, 5, 6, 7, None],
            'learning_rate'    : [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
            'max_leaf_nodes'   : [15, 31, 63, 127, None],
            'min_samples_leaf' : [5, 10, 20, 30],
            'l2_regularization': [0, 0.01, 0.1, 0.5, 1.0],
        }
    ),
}

def neg_rmse(est, X, y):
    return -np.sqrt(mean_squared_error(y, est.predict(X)))

best_models  = {}
hp_results   = {}

for name, (est, params) in search_configs.items():
    print(f'\n🔍 {name} 하이퍼파라미터 탐색 중... ({N_ITER} iterations × {HP_FOLDS} folds)')
    t0 = time.time()

    rs = RandomizedSearchCV(
        est, param_distributions=params,
        n_iter=N_ITER, scoring=neg_rmse,
        cv=HP_FOLDS, random_state=SEED,
        n_jobs=-1, verbose=0,
        return_train_score=True
    )
    rs.fit(X.values, y)

    best_models[name] = rs.best_estimator_
    hp_results[name]  = rs
    elapsed = time.time() - t0

    print(f'  ✅ Best CV RMSE : {-rs.best_score_:.5f}  |  시간: {elapsed:.1f}s')
    print(f'  🏆 Best params  : {rs.best_params_}')

In [ ]:
# ─── HP 탐색 결과 시각화
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.suptitle('🎛️ Hyperparameter Search — CV RMSE per Iteration')
axes = axes.flatten()

for ax, (name, rs) in zip(axes, hp_results.items()):
    color = MODEL_COLORS[name]
    cv_df = pd.DataFrame(rs.cv_results_)
    rmse  = -cv_df['mean_test_score']
    err   = cv_df['std_test_score']

    # 반복 순서대로
    x = np.arange(len(rmse))
    ax.bar(x, rmse, color=color, alpha=0.6, edgecolor='none', label='CV RMSE')
    ax.fill_between(x, rmse-err, rmse+err, color=color, alpha=0.2)

    best  = rmse.min()
    best_x= rmse.argmin()
    ax.axhline(best, color='white', ls='--', lw=1.5, label=f'Best={best:.5f}')
    ax.scatter([best_x], [best], color='white', s=100, zorder=5)
    ax.annotate(f'Best\n{best:.4f}', (best_x, best),
                textcoords='offset points', xytext=(8, 8),
                color='white', fontsize=9)

    ax.set_title(f'{name}')
    ax.set_xlabel('Search Iteration')
    ax.set_ylabel('CV RMSE')
    ax.legend()

plt.tight_layout()
plt.savefig('plot_07_hp_search.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── 최적 파라미터 요약 표 출력
print('='*70)
print('  최적 하이퍼파라미터 및 CV RMSE 요약')
print('='*70)
rows = []
for name, rs in hp_results.items():
    row = {'Model': name, 'CV_RMSE': round(-rs.best_score_, 5)}
    row.update(rs.best_params_)
    rows.append(row)
    print(f'\n  [{name}]  CV RMSE = {-rs.best_score_:.5f}')
    for k, v in rs.best_params_.items():
        print(f'    {k:30s} = {v}')

hp_summary_df = pd.DataFrame(rows)
hp_summary_df.to_csv('best_hyperparameters.csv', index=False)
print('\n\n✅ best_hyperparameters.csv 저장 완료')

## 🔄 6. KFold 교차검증 학습

In [ ]:
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

X_arr  = X.values
Xt_arr = X_test.values

oof_dict   = {}   # model → oof array
pred_dict  = {}   # model → test pred array
fold_rmse_dict = {}  # model → list of fold RMSEs

for name, best_est in best_models.items():
    print(f'\n{"─"*55}')
    print(f'  🏋️  {name}  ({FOLDS}-Fold CV 학습)')
    print(f'{"─"*55}')

    oof  = np.zeros(len(y))
    pred = np.zeros(len(Xt_arr))
    fold_rmses = []

    for fold_i, (tr_idx, val_idx) in enumerate(kf.split(X_arr, y)):
        Xtr, ytr   = X_arr[tr_idx], y[tr_idx]
        Xval, yval = X_arr[val_idx], y[val_idx]

        m = copy.deepcopy(best_est)
        m.fit(Xtr, ytr)

        oof[val_idx] = m.predict(Xval)
        pred        += m.predict(Xt_arr)

        fold_rmse = np.sqrt(mean_squared_error(yval, oof[val_idx]))
        fold_rmses.append(fold_rmse)
        print(f'  Fold {fold_i+1:2d}/{FOLDS}  RMSE = {fold_rmse:.5f}')

    pred /= FOLDS
    oof_rmse = np.sqrt(mean_squared_error(y, oof))

    oof_dict[name]        = oof
    pred_dict[name]       = pred
    fold_rmse_dict[name]  = fold_rmses

    print(f'\n  ⭐ OOF RMSE   = {oof_rmse:.5f}')
    print(f'     Fold Mean  = {np.mean(fold_rmses):.5f} ± {np.std(fold_rmses):.5f}')

## 📊 7. 학습 결과 시각화

In [ ]:
# ─── Fold별 RMSE 비교 (라인 + 박스)
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.suptitle('📊 KFold CV RMSE — All Models')

# 라인 그래프
for name, rmses in fold_rmse_dict.items():
    x = np.arange(1, len(rmses)+1)
    axes[0].plot(x, rmses, marker='o', lw=2, ms=7,
                 color=MODEL_COLORS[name], label=f'{name} (μ={np.mean(rmses):.4f})')
    axes[0].axhline(np.mean(rmses), ls=':', lw=1, color=MODEL_COLORS[name], alpha=0.6)

axes[0].set_title('Fold RMSE per Fold')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('RMSE')
axes[0].set_xticks(np.arange(1, FOLDS+1))
axes[0].legend()

# 박스 플롯
names  = list(fold_rmse_dict.keys())
data   = list(fold_rmse_dict.values())
colors = [MODEL_COLORS[n] for n in names]
bp = axes[1].boxplot(data, labels=names, patch_artist=True,
                     medianprops={'color':'white','lw':2.5})
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.75)
for w in bp['whiskers']: w.set_color('#8b949e')
for cap in bp['caps']:   cap.set_color('#8b949e')
for flier in bp['fliers']:
    flier.set(marker='o', markerfacecolor='white', markersize=5, alpha=0.5)

axes[1].set_title('RMSE Distribution (Boxplot)')
axes[1].set_ylabel('RMSE')

plt.tight_layout()
plt.savefig('plot_08_fold_rmse.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── OOF Predicted vs Actual
n_models = len(oof_dict)
fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 6))
fig.suptitle('📊 OOF Predicted vs Actual  (log1p space)')

for ax, (name, oof) in zip(axes, oof_dict.items()):
    r = np.sqrt(mean_squared_error(y, oof))
    ax.scatter(y, oof, alpha=0.25, s=8,
               c=MODEL_COLORS[name], edgecolors='none')
    lo, hi = min(y.min(), oof.min()), max(y.max(), oof.max())
    ax.plot([lo, hi], [lo, hi], 'w--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{name}\nOOF RMSE={r:.5f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot_09_oof_scatter.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── 잔차(Residual) 분석
fig, axes = plt.subplots(2, n_models, figsize=(6*n_models, 10))
fig.suptitle('📊 Residual Analysis')

for col, (name, oof) in enumerate(oof_dict.items()):
    res = y - oof
    c   = MODEL_COLORS[name]

    # 잔차 vs 예측값
    axes[0, col].scatter(oof, res, alpha=0.25, s=8, color=c, edgecolors='none')
    axes[0, col].axhline(0, color='white', ls='--', lw=1.5)
    axes[0, col].set_xlabel('Predicted'); axes[0, col].set_ylabel('Residual')
    axes[0, col].set_title(f'{name} — Residuals vs Fitted')

    # 잔차 히스토그램
    axes[1, col].hist(res, bins=60, color=c, alpha=0.8, edgecolor='none')
    axes[1, col].axvline(0, color='white', ls='--', lw=1.5)
    axes[1, col].set_xlabel('Residual'); axes[1, col].set_ylabel('Count')
    axes[1, col].set_title(
        f'Residual Dist  skew={pd.Series(res).skew():.3f}'
    )

plt.tight_layout()
plt.savefig('plot_10_residuals.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── Feature Importance
fig, axes = plt.subplots(1, n_models, figsize=(7*n_models, 9))
fig.suptitle('📊 Top-20 Feature Importances')
TOP = 20

for ax, (name, est) in zip(axes, best_models.items()):
    try:
        imp = est.feature_importances_
    except AttributeError:
        ax.set_title(f'{name}\n(importance N/A)'); continue

    idx = np.argsort(imp)[::-1][:TOP]
    top_f = [FEATURES[i] for i in idx]
    top_v = imp[idx]

    cmap = plt.cm.get_cmap('plasma', TOP)
    bar_colors = [cmap(i/TOP) for i in range(TOP)]

    ax.barh(range(TOP), top_v[::-1], color=bar_colors[::-1], alpha=0.85, edgecolor='none')
    ax.set_yticks(range(TOP))
    ax.set_yticklabels(top_f[::-1], fontsize=9)
    ax.set_xlabel('Importance')
    ax.set_title(f'{name}')

plt.tight_layout()
plt.savefig('plot_11_feature_importance.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🤝 8. 앙상블 & 최종 RMSE 비교

In [ ]:
# ── 개별 모델 OOF RMSE
oof_rmses = {n: np.sqrt(mean_squared_error(y, oof_dict[n])) for n in oof_dict}

# ── Equal-weight 앙상블
eq_oof    = np.mean(list(oof_dict.values()),  axis=0)
eq_pred   = np.mean(list(pred_dict.values()), axis=0)
eq_rmse   = np.sqrt(mean_squared_error(y, eq_oof))

# ── Inverse-RMSE 가중 앙상블
inv_w   = {n: 1.0/v for n,v in oof_rmses.items()}
total_w = sum(inv_w.values())
weights = {n: v/total_w for n,v in inv_w.items()}

wtd_oof  = sum(oof_dict[n]  * weights[n] for n in weights)
wtd_pred = sum(pred_dict[n] * weights[n] for n in weights)
wtd_rmse = np.sqrt(mean_squared_error(y, wtd_oof))

# ── 결과 정리
all_rmses = {**oof_rmses, 'EqualEnsemble': eq_rmse, 'WeightedEnsemble': wtd_rmse}
best_strategy = min(all_rmses, key=all_rmses.get)

print('='*60)
print('  RMSE 종합 비교')
print('='*60)
for k, v in sorted(all_rmses.items(), key=lambda x: x[1]):
    marker = '  ← BEST ✅' if k == best_strategy else ''
    print(f'  {k:25s} : {v:.5f}{marker}')
print('='*60)
print(f'\n앙상블 가중치:')
for n, w in weights.items():
    print(f'  {n}: {w:.4f}')

In [ ]:
# ─── 최종 RMSE 시각화
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('📊 Final RMSE Comparison — All Strategies')

all_names  = list(all_rmses.keys())
all_vals   = list(all_rmses.values())
bar_colors = (
    [MODEL_COLORS[n] for n in oof_rmses.keys()]
    + ['#fbbf24', '#ec4899']
)

# RMSE 막대
bars = axes[0].bar(all_names, all_vals, color=bar_colors, alpha=0.85, edgecolor='none')
best_val = min(all_vals)
axes[0].axhline(best_val, color='white', ls='--', lw=1.5, label=f'Best={best_val:.5f}')
for bar, val, nm in zip(bars, all_vals, all_names):
    star = '⭐' if nm == best_strategy else ''
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.0003,
                 f'{star}{val:.4f}', ha='center', fontsize=10, color='white')
axes[0].set_ylabel('OOF RMSE')
axes[0].set_title('OOF RMSE by Strategy')
axes[0].legend()
axes[0].set_xticklabels(all_names, rotation=25, ha='right')

# 개선율 (worst 대비)
worst = max(all_vals)
improvements = [(worst - v) / worst * 100 for v in all_vals]
axes[1].bar(all_names, improvements, color=bar_colors, alpha=0.85, edgecolor='none')
for i, (nm, imp) in enumerate(zip(all_names, improvements)):
    axes[1].text(i, imp+0.05, f'{imp:.2f}%', ha='center', color='white', fontsize=10)
axes[1].set_ylabel('개선율 (%)')
axes[1].set_title('Relative Improvement vs Worst')
axes[1].set_xticklabels(all_names, rotation=25, ha='right')

plt.tight_layout()
plt.savefig('plot_12_final_rmse.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 💾 9. 제출 파일 생성 & 저장

In [ ]:
# ── 최적 전략 선택
if best_strategy in pred_dict:
    final_log = pred_dict[best_strategy]
elif best_strategy == 'EqualEnsemble':
    final_log = eq_pred
else:
    final_log = wtd_pred

print(f'사용 전략 : {best_strategy}')
print(f'OOF RMSE  : {all_rmses[best_strategy]:.5f}')

# ── 역변환 + 클리핑
final_pred = np.clip(np.expm1(final_log), CAL_MIN, CAL_MAX)

print(f'\n예측 통계 (클리핑 후):')
print(f'  min={final_pred.min():.2f}  max={final_pred.max():.2f}')
print(f'  mean={final_pred.mean():.2f}  median={np.median(final_pred):.2f}')

# ── Submission CSV
submission = pd.DataFrame({
    'ID'             : test_raw['ID'],
    'Calories_Burned': np.round(final_pred, 4)
})
submission.to_csv('submission.csv', index=False)
print('\n✅ submission.csv 저장 완료')
display(submission.head(10))

In [ ]:
# ── OOF 전체 저장
oof_df = pd.DataFrame({'ID': train_raw['ID'],
                       'Actual': train_raw['Calories_Burned']})
for n, oof in oof_dict.items():
    oof_df[f'OOF_{n}'] = np.round(np.expm1(oof), 4)
oof_df['OOF_Equal']    = np.round(np.expm1(eq_oof), 4)
oof_df['OOF_Weighted'] = np.round(np.expm1(wtd_oof), 4)

oof_df.to_csv('oof_predictions.csv', index=False)
print('✅ oof_predictions.csv 저장 완료')
display(oof_df.head())

In [ ]:
# ── 예측 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('📊 Final Prediction Distribution')

# Train actual vs Test predicted
axes[0].hist(train_raw['Calories_Burned'], bins=60,
             alpha=0.6, color='#f97316', label='Train Actual', edgecolor='none')
axes[0].hist(final_pred, bins=60,
             alpha=0.6, color='#22c55e', label='Test Predicted', edgecolor='none')
axes[0].set_title('Train Actual vs Test Predicted')
axes[0].set_xlabel('Calories_Burned')
axes[0].legend()

# OOF 실제 vs 예측 (원본 스케일)
best_oof_orig = np.expm1(oof_dict[best_strategy]) if best_strategy in oof_dict \
                else np.expm1(eq_oof)
axes[1].scatter(train_raw['Calories_Burned'], best_oof_orig,
                alpha=0.3, s=10, color=MODEL_COLORS.get(best_strategy,'#fbbf24'),
                edgecolors='none')
lo, hi = 0, max(train_raw['Calories_Burned'].max(), best_oof_orig.max())
axes[1].plot([lo,hi],[lo,hi],'w--',lw=1.5,label='y=x')
axes[1].set_xlabel('Actual Calories_Burned')
axes[1].set_ylabel('OOF Predicted Calories_Burned')
axes[1].set_title(f'{best_strategy} — Original Scale')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_13_pred_dist.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 📋 10. 최종 요약

In [ ]:
# ─── Summary dashboard
fig = plt.figure(figsize=(20, 10))
fig.suptitle('🏁 Final Summary Dashboard — Calories Prediction Pipeline', fontsize=18)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1) RMSE 랭킹
ax1 = fig.add_subplot(gs[0, 0])
sorted_names = sorted(all_rmses, key=all_rmses.get)
sorted_vals  = [all_rmses[n] for n in sorted_names]
bar_c = ['#ffd700' if n==best_strategy else '#8b949e' for n in sorted_names]
ax1.barh(sorted_names, sorted_vals, color=bar_c, alpha=0.85, edgecolor='none')
for i, (n, v) in enumerate(zip(sorted_names, sorted_vals)):
    ax1.text(v+0.0002, i, f'{v:.4f}', va='center', fontsize=10)
ax1.set_title('RMSE Ranking')
ax1.set_xlabel('OOF RMSE')
ax1.invert_yaxis()

# 2) Fold RMSE 라인
ax2 = fig.add_subplot(gs[0, 1])
for name, rmses in fold_rmse_dict.items():
    ax2.plot(np.arange(1,len(rmses)+1), rmses,
             marker='o', lw=2, ms=5, color=MODEL_COLORS[name], label=name)
ax2.set_title('Fold RMSE per Fold')
ax2.set_xlabel('Fold')
ax2.set_ylabel('RMSE')
ax2.legend(fontsize=9)

# 3) Prediction histogram
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(train_raw['Calories_Burned'], bins=50, alpha=0.6,
         color='#f97316', label='Train', edgecolor='none')
ax3.hist(final_pred, bins=50, alpha=0.6,
         color='#22c55e', label='Test Pred', edgecolor='none')
ax3.set_title('Train vs Test Distribution')
ax3.legend(fontsize=9)

# 4) Best model OOF scatter
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(y, best_oof_orig if hasattr(best_oof_orig,'__len__') else np.expm1(eq_oof),
            alpha=0.25, s=8, color='#a855f7', edgecolors='none')
lo2 = 0; hi2 = train_raw['Calories_Burned'].max()
ax4.plot([lo2,hi2],[lo2,hi2],'w--',lw=1.5)
ax4.set_title(f'OOF: {best_strategy}')
ax4.set_xlabel('Actual'); ax4.set_ylabel('Predicted')

# 5) HP search — best model
ax5 = fig.add_subplot(gs[1, 1])
bm = sorted(oof_rmses, key=oof_rmses.get)[0]
cv_df5 = pd.DataFrame(hp_results[bm].cv_results_)
ax5.scatter(range(len(cv_df5)), -cv_df5['mean_test_score'],
            c=range(len(cv_df5)), cmap='plasma', s=40, edgecolors='none')
ax5.axhline(-hp_results[bm].best_score_, color='white', ls='--', lw=1.5)
ax5.set_title(f'HP Search: {bm}')
ax5.set_xlabel('Iteration'); ax5.set_ylabel('CV RMSE')

# 6) 텍스트 요약
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
lines = [
    '📊 PIPELINE SUMMARY',
    '',
    f'  Train  : {len(train_raw):,} samples',
    f'  Test   : {len(test_raw):,} samples',
    f'  Features: {len(FEATURES)}',
    f'  KFolds : {FOLDS}',
    f'  HP iter: {N_ITER}',
    '',
    '  RMSE Rankings:',
]
for k in sorted(all_rmses, key=all_rmses.get):
    lines.append(f'  {"★" if k==best_strategy else " "} {k}: {all_rmses[k]:.5f}')
lines += ['', f'  Best: {best_strategy}']
ax6.text(0.05, 0.95, '\n'.join(lines), transform=ax6.transAxes,
         fontsize=11, verticalalignment='top', family='monospace',
         color='#e6edf3', bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

plt.savefig('plot_14_dashboard.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ─── 전체 결과 RMSE CSV 저장
results_df = pd.DataFrame([
    {'Strategy': k, 'OOF_RMSE': round(v, 6), 'Is_Best': k==best_strategy}
    for k, v in sorted(all_rmses.items(), key=lambda x: x[1])
])
results_df.to_csv('rmse_results.csv', index=False)
print('✅ rmse_results.csv 저장 완료')
display(results_df)

print('\n' + '='*60)
print('  📂 생성된 파일 목록')
print('='*60)
files = [
    'submission.csv            → 최종 테스트 예측 (제출용)',
    'oof_predictions.csv       → OOF 예측 (검증용)',
    'rmse_results.csv          → 모델별 RMSE 결과',
    'best_hyperparameters.csv  → 최적 하이퍼파라미터',
    'plot_01_target_dist.png   → 타겟 분포',
    'plot_02_num_distributions.png → 수치형 피처 분포',
    'plot_03_categorical.png   → 범주형 분석',
    'plot_04_scatter.png       → 피처 vs 타겟',
    'plot_05_heatmap.png       → 상관관계 히트맵',
    'plot_06_feature_corr.png  → 엔지니어링 후 상관관계',
    'plot_07_hp_search.png     → 하이퍼파라미터 탐색',
    'plot_08_fold_rmse.png     → Fold별 RMSE',
    'plot_09_oof_scatter.png   → OOF 산점도',
    'plot_10_residuals.png     → 잔차 분석',
    'plot_11_feature_importance.png → 피처 중요도',
    'plot_12_final_rmse.png    → 최종 RMSE 비교',
    'plot_13_pred_dist.png     → 예측 분포',
    'plot_14_dashboard.png     → 종합 대시보드',
]
for f in files:
    print(f'  {f}')

In [ ]:
# Colab에서 파일 다운로드
try:
    from google.colab import files
    for fname in ['submission.csv','oof_predictions.csv','rmse_results.csv','best_hyperparameters.csv']:
        files.download(fname)
    print('✅ 파일 다운로드 시작됨')
except ImportError:
    print('ℹ️  Colab 외 환경 — 파일은 현재 디렉토리에 저장됨')